# Wind Atlas quickstart — windowed read (live)

Download the **mean wind speed at 100 m** for a small bounding box from the
Global Wind Atlas and read it back. The layer is a 13.9 GB global Cloud-Optimized
GeoTIFF, but the backend reads **only the bbox window** over `/vsicurl/`, so a
small area returns in seconds — no whole-file download.

_Runs live against public figshare (no credentials)._


In [ ]:
from earthlens.core import EarthLens

## Download one windowed subset

A ~0.5° box near DTU (Denmark). `download()` returns the written GeoTIFF path(s).


In [ ]:
paths = EarthLens(
    data_source="solar-wind-atlas",
    variables=["wind_100m"],
    lat_lim=[55.0, 55.5],
    lon_lim=[12.0, 12.5],
    path="swa_out",
).download(progress_bar=False)
paths

## Read the result back with pyramids

The output is an EPSG:4326 GeoTIFF of mean wind speed in m/s.


In [ ]:
import numpy as np
from pyramids.dataset import Dataset

ds = Dataset.read_file(paths[0])
arr = np.asarray(ds.read_array(), dtype='float64')
finite = arr[np.isfinite(arr)]
print('grid     :', ds.columns, 'x', ds.rows, 'EPSG', ds.epsg)
print(
    f'wind m/s : min {finite.min():.2f}  mean {finite.mean():.2f}  max {finite.max():.2f}'
)

## Why this is fast

`download()` opens the remote COG over `/vsicurl/` and reads only the tiles
overlapping the bbox via pyramids `Dataset.crop(bbox=)` — a few hundred KB, not the
13.9 GB global file. The Global Solar Atlas layers (`ghi`, `dni`, …) instead
download their full ~2.7 GB ZIP once and crop locally, since they are not served
as range-accessible COGs.
